# 02 - Text Preprocessing and TF-IDF
Demonstrates the reusable functions in `src/preprocessing.py`.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.config import load_config, resolve
cfg = load_config()
FIG = resolve(cfg, "figures_dir"); FIG.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", palette="deep")

In [ ]:
from src.preprocessing import clean_text, strip_metadata, tokenize, remove_stopwords, fit_vectorizer
from src.data_loader import load_processed_dataset, split_frames
df = load_processed_dataset(cfg)
sample = df[df.dataset_split == "train"].iloc[3]
print("RAW:\n", sample.document_text[:700])

## Why remove headers, footers and quotes?
Usenet headers contain fields such as `Newsgroups:`, `Subject:` and `Organization:`; quoted replies repeat earlier posts. Both can reveal the label without the *content* of the message being understood. The primary model therefore uses text with `headers, footers, quotes` removed (scikit-learn's `remove=` option); the effect is quantified by `python -m src.train --compare-metadata` on the validation split.

In [ ]:
print("WITHOUT METADATA (scikit-learn removal):\n", sample.text_no_metadata[:500])
print("\nCLEANED:\n", sample.cleaned_text[:500])
print("\nTOKENS:", tokenize(sample.cleaned_text)[:25])
print("\nTOKENS WITHOUT STOPWORDS:", remove_stopwords(tokenize(sample.cleaned_text))[:25])

In [ ]:
# The heuristic stripper used for documents pasted into the app
post = "From: a@b.com\nSubject: Re: engines\nNewsgroups: rec.autos\n\nJohn wrote:\n> old text\nNew spark plugs fixed it.\n-- \nJohn"
print(strip_metadata(post))

## TF-IDF (fit on training documents only)

In [ ]:
parts = split_frames(df)
vec = fit_vectorizer(parts["train"].cleaned_text, cfg["tfidf"])
X_train = vec.transform(parts["train"].cleaned_text)
print("Shape:", X_train.shape, "| dtype:", X_train.dtype, "| sparse nnz:", X_train.nnz)
dense_gb = X_train.shape[0] * X_train.shape[1] * 4 / 1e9
print(f"Sparse size ~{(X_train.data.nbytes + X_train.indices.nbytes + X_train.indptr.nbytes)/1e6:.1f} MB vs dense ~{dense_gb:.2f} GB")
print("Density: %.4f%%" % (100 * X_train.nnz / (X_train.shape[0] * X_train.shape[1])))
names = vec.get_feature_names_out()
print("Unigrams:", sum(" " not in n for n in names), "| bigrams:", sum(" " in n for n in names))
print("Sample features:", list(names[:15]))

In [ ]:
idf = pd.Series(vec.idf_, index=names).sort_values()
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
idf.head(15).iloc[::-1].plot.barh(ax=axes[0], color="#3949ab"); axes[0].set_title("Lowest IDF (most common terms)"); axes[0].set_xlabel("IDF")
idf.tail(15).plot.barh(ax=axes[1], color="#e65100"); axes[1].set_title("Highest IDF (rarest terms)"); axes[1].set_xlabel("IDF")
plt.tight_layout(); fig.savefig(FIG / "tfidf_idf_extremes.png", dpi=150); plt.show()